# Scaling Law Analysis: AIC/BIC Model Selection

Compares logarithmic vs linear vs power-law models for L_min(ξ_data). Paper: ΔBIC=8.2.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## Model Comparison via AIC/BIC

In [ ]:
from src.scaling.scaling_law_fitter import ScalingLawFitter
from src.scaling.fss_analysis import AICModelSelector
import numpy as np

xi_vals = np.array([5.0, 15.0, 50.0, 100.0, 200.0])
# True logarithmic L_min
rng = np.random.default_rng(42)
l_min = 8.0 * np.log(xi_vals) + rng.normal(0, 0.5, len(xi_vals))

# Fit three models
fitter = ScalingLawFitter()
log_res   = fitter.fit_logarithmic(xi_vals, l_min)
pow_res   = fitter.fit_power_law(xi_vals, l_min)
lin_res   = fitter.fit_linear(xi_vals, l_min)

print(f"Logarithmic: R²={log_res.r2:.4f}, RMSE={log_res.rmse:.3f}")
print(f"Power-law:   R²={pow_res.r2:.4f}, RMSE={pow_res.rmse:.3f}")
print(f"Linear:      R²={lin_res.r2:.4f}, RMSE={lin_res.rmse:.3f}")

# AIC comparison
selector = AICModelSelector()
models = selector.select(xi_vals, l_min)
print("\nAIC comparison:")
for name, res in models.items():
    print(f"  {name:<12}: AIC={res.aic:.2f}")
best = min(models, key=lambda k: models[k].aic)
print(f"Best: {best} (paper expects logarithmic)")
